<a href="https://colab.research.google.com/github/HakumenWorld/data-science-2026/blob/main/Pertemuan10_Febi_Kristiana_240401010231.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aktivitas Hands-on: Customer Churn
**Mata Kuliah:** Pengantar Data Science  
**Pertemuan:** 10  
**Nama:** Febi Kristiana  
**NIM:** 240401010231

Notebook ini disusun mengikuti bagian **Aktivitas Hands-on** pada modul pertemuan 10.  
Jalankan sel secara berurutan dari atas ke bawah di Google Colab.

## Aktivitas Hands-on — Customer Churn
Dataset yang digunakan adalah **Telco Customer Churn** dengan target `Churn (Yes/No)`.
Dataset bersifat *imbalanced*, sehingga model utama menggunakan
`RandomForestClassifier(class_weight="balanced")`.

> **Tambahan agar cocok di Google Colab:** unggah file dataset dengan nama `telco_churn.csv`.

## Persiapan Colab — Upload `telco_churn.csv`

In [1]:
import os
import pandas as pd

file_path = "telco_churn.csv"

if not os.path.exists(file_path):
    try:
        from google.colab import files
        print("Silakan upload file telco_churn.csv")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        uploaded_name = next(iter(uploaded))
        file_path = uploaded_name
    except ImportError:
        raise FileNotFoundError(
            "File telco_churn.csv belum tersedia. "
            "Jika dijalankan di luar Colab, letakkan file di folder notebook."
        )

print("Dataset yang digunakan:", file_path)

Silakan upload file telco_churn.csv


Saving Telco-Customer-Churn.csv to Telco-Customer-Churn.csv
Dataset yang digunakan: Telco-Customer-Churn.csv


## Langkah 1 — Muat dan Eksplorasi Data

In [2]:
df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nTipe data:")
print(df.dtypes)
print("\nProporsi Churn:")
print(df["Churn"].value_counts(normalize=True).round(4))

display(df.head())

Shape: (7043, 21)

Tipe data:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Proporsi Churn:
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Langkah 2 — Preprocessing

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

data = df.copy()

# Pada dataset Telco Customer Churn yang umum, TotalCharges sering terbaca sebagai object.
if "TotalCharges" in data.columns:
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Target: Yes=1 (churn), No=0
y = data["Churn"].map({"Yes": 1, "No": 0})
if y.isna().any():
    raise ValueError("Nilai target Churn tidak hanya berisi 'Yes' dan 'No'.")

X = data.drop(columns=["Churn"])

# Hapus identifier pelanggan bila ada
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

# Isi missing value numerik dengan median dan kategorikal dengan modus
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        mode_val = X[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if len(mode_val) else "Unknown"
        X[col] = X[col].fillna(fill_val)

# One-Hot Encoding fitur kategorikal
X = pd.get_dummies(X, drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Shape X setelah encoding:", X.shape)
print("Data latih:", X_tr.shape, "| Data uji:", X_te.shape)
print("Proporsi churn data latih:", round(y_tr.mean(), 4))
print("Proporsi churn data uji  :", round(y_te.mean(), 4))

Shape X setelah encoding: (7043, 30)
Data latih: (5634, 30) | Data uji: (1409, 30)
Proporsi churn data latih: 0.2654
Proporsi churn data uji  : 0.2654


## Langkah 3 — Latih Model

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_tr, y_tr)

print("Model Random Forest selesai dilatih.")

Model Random Forest selesai dilatih.


## Langkah 4 — Evaluasi

In [5]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)

pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print("Classification Report:")
print(classification_report(y_te, pred, target_names=["No Churn", "Churn"]))
print(f"ROC-AUC: {roc_auc_score(y_te, proba):.3f}")

precision_churn = precision_score(y_te, pred)
recall_churn = recall_score(y_te, pred)
f1_churn = f1_score(y_te, pred)

print(f"Precision kelas Churn: {precision_churn:.3f}")
print(f"Recall kelas Churn   : {recall_churn:.3f}")
print(f"F1-score kelas Churn : {f1_churn:.3f}")

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.83      0.89      0.86      1035
       Churn       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.825
Precision kelas Churn: 0.630
Recall kelas Churn   : 0.500
F1-score kelas Churn : 0.557


## Langkah 5 — Prediksi Probabilitas dan Simpulkan

In [6]:
# Probabilitas churn untuk data uji
hasil_churn = pd.DataFrame({
    "Aktual": y_te.values,
    "Prediksi": pred,
    "Probabilitas_Churn": proba
}, index=X_te.index)

print("10 pelanggan pada data uji dengan probabilitas churn tertinggi:")
display(hasil_churn.sort_values("Probabilitas_Churn", ascending=False).head(10))

10 pelanggan pada data uji dengan probabilitas churn tertinggi:


,Aktual,Prediksi,Probabilitas_Churn
1731,1,1,1.000000
2194,1,1,0.993333
809,1,1,0.990000
6623,1,1,0.990000
1739,1,1,0.986667
2927,0,1,0.986667
3346,0,1,0.970000
1144,1,1,0.963333
4585,1,1,0.960000
2729,1,1,0.950000


In [7]:
roc_auc = roc_auc_score(y_te, proba)

print(
    f"Kesimpulan: model Random Forest menghasilkan ROC-AUC {roc_auc:.3f}. "
    f"Untuk kelas churn, precision sebesar {precision_churn:.3f}, recall {recall_churn:.3f}, "
    f"dan F1-score {f1_churn:.3f}. "
    "Karena data churn tidak seimbang, evaluasi tidak cukup hanya menggunakan accuracy; "
    "recall dan F1-score kelas churn lebih berguna untuk menilai kemampuan model menemukan "
    "pelanggan yang benar-benar berisiko berhenti. "
    "Probabilitas churn juga dapat digunakan untuk memprioritaskan pelanggan yang perlu "
    "mendapat tindakan retensi lebih dahulu."
)

Kesimpulan: model Random Forest menghasilkan ROC-AUC 0.825. Untuk kelas churn, precision sebesar 0.630, recall 0.500, dan F1-score 0.557. Karena data churn tidak seimbang, evaluasi tidak cukup hanya menggunakan accuracy; recall dan F1-score kelas churn lebih berguna untuk menilai kemampuan model menemukan pelanggan yang benar-benar berisiko berhenti. Probabilitas churn juga dapat digunakan untuk memprioritaskan pelanggan yang perlu mendapat tindakan retensi lebih dahulu.


## Kesimpulan

Dari aktivitas hands-on ini, kita telah mempelajari proses *end-to-end* untuk membangun model *customer churn* menggunakan `RandomForestClassifier`.

### Temuan Utama:
- Data *Telco Customer Churn* bersifat *imbalanced*, dengan proporsi *churn* sekitar 26.54%.
- Model Random Forest mencapai ROC-AUC 0.825.
- **Precision kelas Churn** sebesar 0.630, menunjukkan 63% dari pelanggan yang diprediksi *churn* memang benar-benar *churn*.
- **Recall kelas Churn** sebesar 0.500, menunjukkan model dapat mengidentifikasi 50% dari total pelanggan yang sebenarnya akan *churn*.
- F1-score kelas Churn sebesar 0.557, memberikan gambaran keseimbangan antara precision dan recall.

### Keterbatasan dan Pertanyaan:
- **Keterbatasan Data**: Hanya menggunakan fitur-fitur yang tersedia. Mungkin ada faktor eksternal lain yang tidak tercakup dalam dataset (misalnya, strategi pesaing, perubahan ekonomi) yang dapat mempengaruhi *churn*.
- **Keterbatasan Model**: Meskipun Random Forest adalah model yang kuat, masih ada ruang untuk eksplorasi model lain atau *tuning hyperparameter* lebih lanjut untuk meningkatkan kinerja.
- **Interpretasi Model**: Meskipun penting untuk memprediksi *churn*, memahami mengapa pelanggan *churn* juga krusial. Analisis *feature importance* atau *Shapley values* dapat memberikan wawasan lebih lanjut.
- **Tindakan Retensi**: Bagaimana probabilitas *churn* dapat diterjemahkan menjadi strategi retensi yang efektif? Apakah ada ambang batas probabilitas tertentu yang harus menjadi pemicu tindakan?
- **Monitoring**: Seberapa sering model perlu dilatih ulang untuk mempertahankan akurasinya seiring waktu, mengingat perilaku pelanggan bisa berubah?